# Буткемп, день 5 — point-in-time фичи из L2 трёх бирж и набор инвариантов, который умеет падать

Задание дня: превратить два дня L2-перпетуалов Bybit, OKX и Binance в point-in-time feature frame
и доказать его корректность набором инвариантов, способным провалиться. Слайд перечисляет пять
результатов — book builder по трём биржам, набор инвариантов, результаты fault injection, feature
frame и отчёт — и четыре обязательных гейта приёмки: (a) корректная книга (восстановленный стакан
совпадает с эталоном), (b) отсутствие look-ahead (проверка усечением входа), (c) упорядоченность по
sequence id внутри потока и as-of джойны по времени приёма между биржами, (d) воспроизводимость
(одинаковые хеши при повторном прогоне и пересборка одной командой). Любой провал гейта означает
незачёт независимо от остального, поэтому гейты ниже показаны числами, а не декларацией.

Выданы четыре файла: три нормализованные выгрузки L2 по USDT-перпетуалам за 03:00 1 июля —
03:00 3 июля 2026 (MSK) и эталон top-of-book на 100-мс сетке. Природу эталона фиксируем сразу,
до сверки: в гейте (a) показано, что его моменты времени в проверенном окне — это моменты сообщений тех же
дампов, а цены и размеры лучших уровней выводятся из них; поэтому сверка с ним — контроль
против эталонной реализации организаторов, а не независимое подтверждение.

| файл | биржа | символы | строк |
|---|---|---|---|
| [`bybit_l2_20260701_20260703.parquet`](../Downloads/bybit_l2_20260701_20260703.parquet) | Bybit | BTCUSDT, ETHUSDT | 134 145 117 |
| [`okex_l2_20260701_20260703.parquet`](../Downloads/okex_l2_20260701_20260703.parquet) | OKX | BTC-USDT-SWAP, ETH-USDT-SWAP | 406 823 347 |
| [`binance_l2_20260701_20260703.parquet`](../Downloads/binance_l2_20260701_20260703.parquet) | Binance | BTCUSDT, ETHUSDT | 692 715 519 |
| [`l2_executable_quotes_100ms_10000.parquet`](../Downloads/l2_executable_quotes_100ms_10000.parquet) | эталон | BTC, ETH | 10 244 979 |

Схема у трёх бирж одна: `entry_type=1` — снапшот (полный срез), `entry_type=2` — инкремент
(изменение уровня), `entry_type=3` — heartbeat, который в стакан не применяется; `_feed` — номер
подфида, у каждого свой поток снапшотов и инкрементов; `side` 1 — bid, −1 — ask; `amount = 0`
снимает уровень, `amount > 0` задаёт **абсолютный** размер (не дельту). Тонкость, определяющая
всю реконструкцию: `event_time` — это **наше** время приёма гейтом, а `exchange_time` — биржевое,
поэтому кросс-биржевые сопоставления делаются только по `event_time`.

Почти 1.23 млрд строк за разумное время построчно не пройти. Поэтому книга
реконструируется на окне (пара переменных `WINDOW_START`, `WINDOW_HOURS`), а по полному периоду идут
только агрегаты — счётчики разрывов, сбросов и латентности; их duckdb считает за секунды-минуты.
Проверка от этого не становится слабее, просто у каждой задачи своя область: агрегатные инварианты считаются на всех 1.23 млрд
строк, дорогие построчные (реплей, prefix-тест, инъекции) — на окне, и в тексте сказано, где какая
область.

In [1]:
import duckdb, numpy as np, pandas as pd, hashlib, os, time, gc
from sortedcontainers import SortedDict

# Сборщик накапливает длинные списки снимков и состояний и не создаёт циклов,
# поэтому автоматический GC только тратит время на обход миллионов объектов.
# На результат это не влияет: детерминизм задан порядком строк
# (event_time, file_row_number), отключение GC меняет только скорость.
gc.disable()

DATA = '/Users/b.kotov/Downloads'
SRC = {
    'bybit':   (f'{DATA}/bybit_l2_20260701_20260703.parquet',   'BTCUSDT'),
    'okx':     (f'{DATA}/okex_l2_20260701_20260703.parquet',    'BTC-USDT-SWAP'),
    'binance': (f'{DATA}/binance_l2_20260701_20260703.parquet', 'BTCUSDT'),
}
REF = f'{DATA}/l2_executable_quotes_100ms_10000.parquet'

WINDOW_START = '2026-07-01 09:32:00+03'   # сразу после первого снапшота каждой биржи
WINDOW_HOURS = 4.0                        # окно построения книги
PREFIX_MIN   = 30                         # окно и набор точек усечения (минут)
N_CUTS       = 24                         # точек усечения в prefix-тесте
PAUSE_S      = 1.0                        # пауза приёма, ниже которой скачок номера — не потеря
SEED         = 20260705

# Тик и лот выводятся из данных (ячейка разведки) и сверяются с этими значениями.
# Лот — самая крупная степень десяти, кратной которой оказываются все размеры окна.
TICK = {'bybit': 0.1, 'okx': 0.1, 'binance': 0.1}
LOT  = {'bybit': 1e-3, 'okx': 1e-4, 'binance': 1e-3}
# Снапшотная глубина базы — предел, до которого у нас есть исходные уровни и снятия.
SNAPDEPTH = {'bybit': 50, 'okx': 400, 'binance': 100}
MAX_SIZE  = 1e6      # лотов = 1000 BTC (Bybit/Binance) либо 100 BTC (OKX). В окне
                     # медианный размер уровня 0.14–0.53 лота, максимальный 89–979: кап
                     # выше максимума на три порядка и срабатывает только на явную порчу
BAND_PCT  = 0.05     # рабочий набор: уровни дальше ±5 % от середины снапшота не реконструируются.
                     # Оценка полосы по прогону окна: Bybit и OKX не выходят за неё ни строкой
                     # (invalid=0), у Binance отсекается 2 824 537 строк — все глубже снапшотной базы

W0 = pd.Timestamp(WINDOW_START)
W1 = W0 + pd.Timedelta(hours=WINDOW_HOURS)

con = duckdb.connect()
con.execute("SET threads=6")
con.execute("SET memory_limit='8GB'")
con.execute("SET temp_directory='/tmp/duckdb_tmp'")
con.execute("SET max_temp_directory_size='6GB'")
print('duckdb', duckdb.__version__, '| окно', W0, '->', W1)

duckdb 1.5.6 | окно 2026-07-01 09:32:00+03:00 -> 2026-07-01 13:32:00+03:00


## Разведка данных

Фиксируем схему и то, чем биржи отличаются. Отличаются биржи в первую очередь набором подфидов: у Bybit и OKX это
1/2/3, у Binance — 1/2/3/**4**. Подфид 4 у Binance несёт только снятия уровней (`amount=0`) и
приходит редко, но именно он закрывает удаления, которых нет в основном потоке; без него в
top-of-book появляются ложные crossed-состояния (ниже это подкреплено числом). Порядок применения
задаётся физической позицией строки: `ORDER BY event_time, file_row_number`, где `file_row_number`
берётся из `read_parquet(..., file_row_number=true)`; сортировать по `sequence_no` нельзя — у feed1
и feed2 это разные пространства нумерации, и сортировка перемешает две независимые
последовательности.

In [2]:
print(con.execute("DESCRIBE SELECT * FROM read_parquet(?)", [SRC['bybit'][0]]).df()
      [['column_name', 'column_type']].to_string(index=False))

print()
rows = []
for name, (path, sym) in SRC.items():
    d = con.execute("""
        SELECT entry_type, _feed, count(*) AS n, count(sequence_no) AS n_with_seq,
               sum(CASE WHEN amount<=0 THEN 1 ELSE 0 END) AS n_delete
        FROM read_parquet(?) WHERE symbol=? AND event_time>? AND event_time<=?
        GROUP BY 1,2 ORDER BY 1,2""", [path, sym, W0, W1]).df()
    d.insert(0, 'venue', name)
    rows.append(d)
print('строки по типу и подфиду в окне:')
print(pd.concat(rows, ignore_index=True).to_string(index=False))

print()
snaps = []
for name, (path, sym) in SRC.items():
    d = con.execute("""
        SELECT event_time, _feed, sequence_no, count(*) AS n,
               count(*) FILTER (WHERE side=1) AS n_bid, count(*) FILTER (WHERE side=-1) AS n_ask
        FROM read_parquet(?) WHERE symbol=? AND entry_type=1
        GROUP BY 1,2,3 ORDER BY 1""", [path, sym]).df()
    snaps.append(d.assign(venue=name))
print('все снапшоты за период (они лежат только в начале файла — это задаёт область реконструкции):')
print(pd.concat(snaps, ignore_index=True).to_string(index=False))

          column_name              column_type
               symbol                  VARCHAR
           event_time TIMESTAMP WITH TIME ZONE
     event_time_nanos                  INTEGER
         insertion_no                   BIGINT
        exchange_time TIMESTAMP WITH TIME ZONE
  exchange_time_nanos                  INTEGER
          sequence_no                   BIGINT
                 side                  INTEGER
                price                   DOUBLE
               amount                   DOUBLE
           entry_type                  INTEGER
        entry_markers                  INTEGER
      _insertion_time TIMESTAMP WITH TIME ZONE
_insertion_time_nanos                  INTEGER
             _gate_id                  INTEGER
                _feed                  INTEGER
       _feed_l2_depth                  INTEGER
             exchange                  VARCHAR



строки по типу и подфиду в окне:
  venue  entry_type  _feed        n  n_with_seq  n_delete
  bybit           2      1  3757935     3757935  899967.0
  bybit           2      2   326421      326421   39546.0
  bybit           2      3    45633           0   45633.0
    okx           2      1 14755334    14755334 2450298.0
    okx           2      2   186969      186969   16645.0
    okx           2      3   114592           0  114592.0
binance           2      1 19708880    19708880 1416608.0
binance           2      2  4914680     4914680   80428.0
binance           2      3   122411           0  122411.0
binance           2      4      633         633     633.0



все снапшоты за период (они лежат только в начале файла — это задаёт область реконструкции):
                      event_time  _feed    sequence_no   n  n_bid  n_ask   venue
2026-07-01 09:31:05.480956+03:00      1   646554928143 100     50     50   bybit
2026-07-01 21:10:24.904450+03:00      2   647957944924   2      1      1   bybit
2026-07-01 21:10:24.908689+03:00      1   647957933170 100     50     50   bybit
2026-07-01 22:02:45.115923+03:00      1   648069606195 100     50     50   bybit
2026-07-01 22:50:12.300959+03:00      1   648168673459 100     50     50   bybit
2026-07-01 22:58:32.519056+03:00      1   648186217080 100     50     50   bybit
2026-07-01 09:28:26.729801+03:00      2   327425058537   2      1      1     okx
2026-07-01 09:28:26.753905+03:00      1   327425058537 800    400    400     okx
2026-07-01 09:29:21.090047+03:00      1 10937614462375 200    100    100 binance


## Цены и размеры — целые тики и лоты

Задание требует целые тики и int64-время, и это не формальность. В дампе цена хранится `DOUBLE`, а
`float64` не годится даже для сложения: `0.1 + 0.2` даёт `0.30000000000000004`, а шаг представления
`np.spacing(58603.45)` — около $7.3 \times 10^{-12}$. При миллионах операций ошибка накапливается, и
сравнение цен `==` делается недетерминированным. Поэтому цены переводятся в целые тики функцией
`round(p / tick)`, а размеры остаются в лотах. Тик и лот не назначаются, а выводятся из данных:
тик — минимальный положительный зазор между различными ценами окна; лот — самая крупная степень
десяти, кратной которой оказываются все размеры (выведенное значение затем сверяется с конфигурацией).

In [3]:
def discover_grid(path, sym, t0, t1):
    px = con.execute("""SELECT DISTINCT price FROM read_parquet(?)
        WHERE symbol=? AND entry_type=2 AND event_time>? AND event_time<=? AND price IS NOT NULL""",
        [path, sym, t0, t1]).df()['price'].to_numpy()
    px = np.sort(px[px > 1.0])
    dif = np.diff(px); dif = dif[dif > 1e-9]
    tick = float(np.round(dif.min(), 8))
    am = con.execute("""SELECT DISTINCT amount FROM read_parquet(?)
        WHERE symbol=? AND entry_type=2 AND event_time>? AND event_time<=? AND amount>0""",
        [path, sym, t0, t1]).df()['amount'].to_numpy()
    lot = None
    for e in range(1, 7):
        L = 10.0 ** -e
        if np.all(np.abs(am / L - np.round(am / L)) < 1e-6):
            lot = L
            break                      # самая крупная подходящая степень — и есть шаг размеров
    return tick, lot, len(px), len(am)

print('выведенные тик/лот против конфигурации:')
for name, (path, sym) in SRC.items():
    tick, lot, npx, nam = discover_grid(path, sym, W0, W0 + pd.Timedelta(minutes=20))
    agree = 'совпадает' if (tick == TICK[name] and lot == LOT[name]) else 'РАСХОЖДЕНИЕ'
    print(f'{name:<8} тик={tick:<6} (в конфиге {TICK[name]}) лот={lot:<7} (в конфиге {LOT[name]}) '
          f'различных цен={npx:>6} различных размеров={nam:>7} — {agree}')

print()
print('почему float64 не годится:')
print('  0.1 + 0.2            =', repr(0.1 + 0.2))
print('  np.spacing(58603.45) =', np.spacing(58603.45))
print('  float: 586.03 + 57017.42 - 57017.42 =', repr(586.03 + 57017.42 - 57017.42))
print('  int:   58603  + 5701742   - 5701742 =', 58603 + 5701742 - 5701742)

выведенные тик/лот против конфигурации:
bybit    тик=0.1    (в конфиге 0.1) лот=0.001   (в конфиге 0.001) различных цен=  3200 различных размеров=  13897 — совпадает


okx      тик=0.1    (в конфиге 0.1) лот=0.0001  (в конфиге 0.0001) различных цен=  4014 различных размеров=  69112 — совпадает


binance  тик=0.1    (в конфиге 0.1) лот=0.001   (в конфиге 0.001) различных цен= 33251 различных размеров=  46452 — совпадает

почему float64 не годится:
  0.1 + 0.2            = 0.30000000000000004
  np.spacing(58603.45) = 7.275957614183426e-12
  float: 586.03 + 57017.42 - 57017.42 = 586.0299999999988
  int:   58603  + 5701742   - 5701742 = 58603


## Задержка приёма

Задержку приёма считаем с наносекундной точностью стороны приёма:
$\ell = \big(\mathrm{epoch}(T^{recv}) + n^{recv}/10^9\big) - \big(\mathrm{epoch}(T^{exch}) + n^{exch}/10^9\big)$,
где `event_time`/`event_time_nanos` — наши штампы, `exchange_time`/`exchange_time_nanos` —
биржевые. Это число нужно, чтобы оценить разброс приёма между биржами и убедиться, что штампы не
из будущего: отрицательная задержка означала бы рассинхронизацию или ошибку выгрузки.

In [4]:
lat = []
for name, (path, sym) in SRC.items():
    r = con.execute("""
        SELECT min(l) AS mn, quantile_cont(l, 0.5) AS p50, quantile_cont(l, 0.99) AS p99, max(l) AS mx,
               sum(CASE WHEN l < 0 THEN 1 ELSE 0 END) AS n_neg
        FROM (SELECT epoch(event_time) + event_time_nanos/1e9
                     - epoch(exchange_time) - exchange_time_nanos/1e9 AS l
              FROM read_parquet(?) WHERE symbol=? AND entry_type=2 AND event_time>? AND event_time<=?)
        """, [path, sym, W0, W1]).df().iloc[0]
    lat.append(dict(venue=name, min_ms=round(r.mn*1e3, 3), p50_ms=round(r.p50*1e3, 3),
                    p99_ms=round(r.p99*1e3, 3), max_ms=round(r.mx*1e3, 1), negative=int(r.n_neg)))
print(pd.DataFrame(lat).to_string(index=False))

  venue  min_ms  p50_ms  p99_ms  max_ms  negative
  bybit  35.823  37.304  45.742    91.3         0
    okx  21.402  23.740  31.678   288.0         0
binance   1.062   2.234   5.225   163.7         0


## Порог разрыва калибруется по данным

Разрыв — это пропуск сообщений в нумерации подфида. Шаг `sequence_no` не равен единице: биржа пакует
в один номер целую дельту, поэтому шаг болтается от единиц до десятков тысяч, и единая константа
не годится — у Bybit p99.99 шага подфида 2 около 41 тыс., у OKX около 6 тыс. Второй признак — пауза
приёма: настоящая потеря отзывается паузой (пока фид молчал, биржа ушла вперёд), а большой шаг при
нормальном каденсе — это просто толстая дельта. Порог для каждого подфида равен максимальному шагу,
наблюдённому при паузе приёма не больше секунды, плюс единица; такой порог не даёт ложных
срабатываний по построению, потому что всё, что при короткой паузе его не превысило, физически
неотличимо от плотной дельты.

Разрывы, сбросы и дубликаты считаются по полному периоду и раскладываются по бирже и часу — это
требуемый в задании счётчик брешей по venue × hour.

In [5]:
GAP_THR = {}
cal, hourly = [], []
for name, (path, sym) in SRC.items():
    df = con.execute("""
        WITH u AS (   -- одно сообщение = один sequence_no; строк в сообщении десятки
          SELECT _feed f, sequence_no s, min(file_row_number) frn, min(event_time) t
          FROM read_parquet(?, file_row_number=true)
          WHERE symbol=? AND entry_type=2 AND _feed IN (1,2) AND sequence_no IS NOT NULL
          GROUP BY 1,2),
        m AS (SELECT f, s, s - lag(s) OVER (PARTITION BY f ORDER BY frn) d,
                     epoch(t - lag(t) OVER (PARTITION BY f ORDER BY frn)) dt FROM u)
        SELECT f, count(*) n, quantile_cont(d,0.5) p50, quantile_cont(d,0.999) p999,
               max(d) FILTER (WHERE d>0 AND dt<=?) mx
        FROM m GROUP BY 1 ORDER BY 1""", [path, sym, PAUSE_S]).df()
    GAP_THR[name] = {int(r.f): int(r.mx) + 1 for r in df.itertuples(index=False)}
    for r in df.itertuples(index=False):
        cal.append(dict(venue=name, feed=int(r.f), msgs=int(r.n), p50=int(r.p50),
                        p999=int(r.p999), max_normal=int(r.mx), threshold=int(r.mx)+1))
    hb = con.execute("""
        WITH u AS (
          SELECT _feed f, sequence_no s, min(file_row_number) frn, min(event_time) t
          FROM read_parquet(?, file_row_number=true)
          WHERE symbol=? AND entry_type=2 AND _feed IN (1,2) AND sequence_no IS NOT NULL
          GROUP BY 1,2),
        m AS (SELECT f, t, s, s - lag(s) OVER (PARTITION BY f ORDER BY frn) d,
                     epoch(t - lag(t) OVER (PARTITION BY f ORDER BY frn)) dt FROM u)
        SELECT strftime(t, '%m-%d %H:00') hr, f AS feed,
               sum(CASE WHEN d<0 THEN 1 ELSE 0 END) resets,
               sum(CASE WHEN d>0 AND d > CASE f WHEN 1 THEN ? WHEN 2 THEN ? END AND dt>?
                        THEN 1 ELSE 0 END) gaps
        FROM m GROUP BY 1,2 HAVING resets>0 OR gaps>0 ORDER BY 1, 2""",
        [path, sym, GAP_THR[name][1], GAP_THR[name][2], PAUSE_S]).df()
    if len(hb):
        hourly.append(hb.assign(venue=name))

print('калибровка порога по подфидам (полный период):')
print(pd.DataFrame(cal).to_string(index=False))
print()
print('разрывы и сбросы по бирже и часу (полный период, калиброванный порог):')
print(pd.concat(hourly, ignore_index=True).to_string(index=False) if hourly else '(брешей нет)')
if hourly:
    H = pd.concat(hourly, ignore_index=True)
    bb = H[H.venue == 'bybit'].groupby('feed')['gaps'].sum()
    print('итого за период: Bybit', int(H.loc[H.venue == 'bybit', 'gaps'].sum()),
          f'(feed1 {int(bb.get(1, 0))}, feed2 {int(bb.get(2, 0))}), OKX',
          int(H.loc[H.venue == 'okx', 'gaps'].sum()), ', Binance',
          int(H.loc[H.venue == 'binance', 'gaps'].sum()))

калибровка порога по подфидам (полный период):
  venue  feed     msgs  p50  p999  max_normal  threshold
  bybit     1  6189202  692  9032       43834      43835
  bybit     2  3013947 1044 33719       59711      59712
    okx     1 13221421   10  2018       10385      10386
    okx     2  3129893   58  3473       63012      63013
binance     1  6421057 2259 21869       55416      55417
binance     2 76879091   22 13951      113833     113834

разрывы и сбросы по бирже и часу (полный период, калиброванный порог):
         hr  feed  resets  gaps   venue
07-01 03:00     2     0.0  21.0   bybit
07-01 04:00     2     0.0   2.0   bybit
07-01 06:00     2     0.0  11.0   bybit
07-01 07:00     2     0.0   1.0   bybit
07-01 08:00     2     0.0  11.0   bybit
07-01 09:00     1     0.0   1.0   bybit
07-01 09:00     2     0.0   3.0   bybit
07-01 10:00     1     0.0   1.0   bybit
07-01 10:00     2     0.0   7.0   bybit
07-01 11:00     2     0.0   4.0   bybit
07-01 12:00     2     0.0   9.0   bybit
07

## Слои обработки

Реконструкция разложена на четыре слоя, каждый проверяется отдельно.

**Parser** читает parquet потоково в порядке `(event_time, file_row_number)` и переводит строки в
целые тики и лоты. Он же ограничивает рабочий набор полосой цен ±5 % вокруг середины снапшота.
Ограничение вынужденное: снапшот задаёт всего 50–400 уровней (доли процента от
цены), а дельты у Binance тянутся к уровням до 8 тыс. по бидам и до 600 тыс. по аскам — это
реальные публикуемые уровни, но восстановить их согласованно нельзя, потому что снятий за
пределами снапшотной базы в дампе нет. Поэтому всё за полосой считается вне рабочего набора,
а число отброшенных строк идёт в счётчик `invalid` (в окне: у Bybit и OKX нули, у Binance
2 824 537); на top-of-book это не влияет, потому что отброшенные уровни
отстоят от середины на десятки процентов.

**Sequencer** — машина состояний `WAIT_SNAPSHOT → SYNCING → LIVE`; из `LIVE` при разрыве или сбросе
базового подфида — в `UNSYNCED`, и обратно в `SYNCING` при следующем снапшоте (`RESYNC`). Снапшот
заменяет книгу целиком, а не сливается с ней. Эмитит только `LIVE` (а `UNSYNCED` — лишь с флагом
непригодности); в `WAIT_SNAPSHOT` и `SYNCING` наружу ничего не идёт. Каждый переход пишется в журнал.

**Book** — venue-agnostic лестница: `SortedDict` для bid (ключ — отрицательный тик, чтобы лучший bid
был первым) и для ask; операция одна — set абсолютного размера либо удаление при `amount <= 0`, плюс
контроль размера уровней.

**Frame** — эмиссия по границе сообщения: состояние публикуется ровно один раз, на конце сообщения,
и содержит только применённые к этому моменту данные. Это и есть point-in-time: строка со временем
$t$ не видит ни одной строки со временем больше $t$.

Граница эмиссии выбрана не зря: сообщение — это один `sequence_no`, внутри которого лежат десятки строк;
внутри сообщения книга проходит промежуточные состояния, и часть из них crossed. Это артефакт
порядка применения внутри сообщения, а не наблюдаемое рынком состояние, поэтому промежуточные
crossed-состояния наружу не отдаются: эмиссия стоит на границе сообщения. Если же crossed
возникает на границе — это уже сигнал поломки реконструкции, а не артефакт порядка: в чистом
прогоне окна таких состояний ноль, а в fault injection они ловятся счётчиком.

In [6]:
def messages_for(path, sym, feeds):
    return tuple(int(x) for x in con.execute("""SELECT DISTINCT _feed FROM read_parquet(?)
        WHERE symbol=? AND entry_type=2 ORDER BY 1""", [path, sym]).df()['_feed'].tolist())


def iter_messages(path, sym, t_lo, t_hi, feeds):
    # Сообщения по границам receive-time в порядке (event_time, file_row_number).
    res = con.execute(
        "SELECT event_time t, _feed f, sequence_no s, side sd, price p, amount a, "
        "       epoch(exchange_time)*1e9 AS ex "
        "FROM read_parquet(?, file_row_number=true) "
        "WHERE symbol=? AND entry_type=2 AND _feed IN " + str(tuple(feeds)) + " "
        "  AND event_time>? AND event_time<=? ORDER BY event_time, file_row_number",
        [path, sym, t_lo, t_hi])
    cur_t, buf = None, []
    while True:
        chunk = res.fetch_df_chunk(100)
        if chunk.shape[0] == 0:
            break
        for row in chunk.itertuples(index=False):
            if cur_t is not None and row.t != cur_t:
                yield cur_t, buf; buf = []
            cur_t = row.t
            buf.append((row.f, row.s, row.sd, row.p, row.a, row.ex))
    if buf:
        yield cur_t, buf


def book_from(df, tick):
    # Снапшот задаёт базу: лестницы строятся с нуля.
    bids, asks = SortedDict(), SortedDict()
    for side, price, amount in df[['side', 'price', 'amount']].itertuples(index=False):
        if amount and amount > 0:
            k = int(round(price / tick))
            (bids if side == 1 else asks)[-k if side == 1 else k] = float(amount)
    return bids, asks


def seed_book(path, sym, t0, feed=1):
    # Последний снапшот не позже t0; если такого нет — самый первый в файле.
    st = con.execute("""SELECT max(event_time) FROM read_parquet(?)
        WHERE symbol=? AND _feed=? AND entry_type=1 AND event_time<=?""", [path, sym, feed, t0]).fetchone()[0]
    if st is None:
        st = con.execute("""SELECT min(event_time) FROM read_parquet(?)
            WHERE symbol=? AND _feed=? AND entry_type=1""", [path, sym, feed]).fetchone()[0]
    df = con.execute("""SELECT event_time, sequence_no, side, price, amount FROM read_parquet(?)
        WHERE symbol=? AND _feed=? AND entry_type=1 AND event_time=?""", [path, sym, feed, st]).df()
    return df, st


def load_snapshots(path, sym, t_after, t_until, feed=1):
    df = con.execute("""SELECT event_time, sequence_no, side, price, amount FROM read_parquet(?)
        WHERE symbol=? AND _feed=? AND entry_type=1 AND event_time>? AND event_time<=?
        ORDER BY event_time""", [path, sym, feed, t_after, t_until]).df()
    return [(t, g) for t, g in df.groupby('event_time')]


def top_depth(sd, n):
    s = 0.0
    for j, (k, v) in enumerate(sd.items()):
        if j >= n:
            break
        s += v
    return s


def frame_row(t_ns, bids, asks, healthy, degraded, tick):
    # Одна строка feature frame: только то, что уже применено к этому моменту.
    # degraded здесь — флаг 0/1 («за историю прогона был хотя бы один гэп вторичного
    # подфида»), тогда как в S['degraded'] лежит число гэпов feed2; в хеш кадра входит флаг.
    if not bids or not asks:
        return None
    pb, qb = -bids.peekitem(0)[0], bids.peekitem(0)[1]
    pa, qa = asks.peekitem(0)[0], asks.peekitem(0)[1]
    mid = (pb + pa) / 2.0
    return dict(t_ns=t_ns, pb=pb, qb=qb, pa=pa, qa=qa,
                spread_bps=(pa - pb) / mid * 1e4 if mid > 0 else np.nan,
                depth_bid=top_depth(bids, 10), depth_ask=top_depth(asks, 10),
                micro=(pa * qb + pb * qa) / (qb + qa) if (qb + qa) > 0 else np.nan,
                obi=(qb - qa) / (qb + qa) if (qb + qa) > 0 else np.nan,
                healthy=int(healthy), degraded=int(degraded), crossed=int(pb >= pa))


print('Parser/Book: iter_messages, book_from, seed_book, load_snapshots, top_depth, frame_row — определены')

Parser/Book: iter_messages, book_from, seed_book, load_snapshots, top_depth, frame_row — определены


In [7]:
LAYERS = 'Parser -> Sequencer -> Book -> Frame'


def reconstruct(msgs, seed_df, seq0, tick, thr, t_from, t_end, resync=(),
                feeds_seq=(1, 2), pause_s=PAUSE_S, band=None):
    """Прогоняет сообщения через слои; возвращает состояния, журнал и счётчики инвариантов."""
    bids, asks = book_from(seed_df, tick)
    last_seq = {f: seq0 for f in feeds_seq}
    last_t   = {f: None for f in feeds_seq}
    seen     = {f: set() for f in feeds_seq}
    ri, state, prev_t = 0, 'SYNCING', None            # база уже загружена -> SYNCING
    S = dict(msgs=0, rows=0, gap=0, reset=0, dup_msg=0, dup_rows=0, orphan=0, invalid=0,
             crossed=0, crossed_healthy=0, blocked=0, degraded=0, resyncs=0, size_cap=0,
             latency_neg=0, time_back=0, max_bid_lvl=0, max_ask_lvl=0)
    trans, gap_ev, F = [], [], []
    for t, rows in msgs:
        if prev_t is not None and t < prev_t:
            S['time_back'] += 1                        # нарушение монотонности порядка прихода
        while ri < len(resync) and t >= resync[ri][0]:
            _, g = resync[ri]
            nb, na = book_from(g, tick)
            bids.clear(); bids.update(nb); asks.clear(); asks.update(na)
            sq = int(g['sequence_no'].iloc[0])
            for f in last_seq:
                last_seq[f] = sq; last_t[f] = None
            trans.append((str(t), 'RESYNC', -1)); state = 'SYNCING'; ri += 1; S['resyncs'] += 1
        for (f, seq, sd, p, a, ex) in rows:
            if p is None or a is None or not np.isfinite(p) or not np.isfinite(a) \
               or (band is not None and not (band[0] < p < band[1])):
                S['invalid'] += 1; continue
            if ex is not None:
                l = t.value / 1e9 - ex / 1e9
                if l < 0: S['latency_neg'] += 1
            if seq is not None and f in last_seq:
                d = int(seq) - last_seq[f]
                if d == 0:
                    S['dup_rows'] += 1                 # та же дельта, ещё одна строка
                else:
                    S['msgs'] += 1
                    pause = None if last_t[f] is None else (t - last_t[f]).total_seconds()
                    if d < 0:
                        if int(seq) in seen[f]:
                            S['dup_msg'] += 1          # дубликат уже виденного сообщения
                        else:
                            S['reset'] += 1            # сброс нумерации
                            trans.append((str(t), 'RESET', f))
                            state = 'UNSYNCED' if f == 1 else state
                            S['degraded'] += (f != 1)
                    elif d > thr.get(f, 10**9) and (pause is not None and pause > pause_s):
                        S['gap'] += 1                  # потеря дельт
                        gap_ev.append((t, f, int(seq), int(d), pause))
                        trans.append((str(t), 'GAP', f))
                        state = 'UNSYNCED' if f == 1 else state
                        S['degraded'] += (f != 1)
                    elif state == 'SYNCING':
                        trans.append((str(t), 'SYNCING->LIVE', f)); state = 'LIVE'
                    last_seq[f] = int(seq); seen[f].add(int(seq)); last_t[f] = t
            k = int(round(p / tick))
            bk = -k if sd == 1 else k
            book = bids if sd == 1 else asks
            if a <= 0:
                if bk not in book:
                    S['orphan'] += 1                  # снятие уровня, которого нет в книге
                book.pop(bk, None)
            else:
                if a > MAX_SIZE:
                    S['size_cap'] += 1                # невозможный размер: flag и clip, не тихая починка
                    a = min(a, MAX_SIZE)
                book[bk] = float(a)
            S['rows'] += 1
        if state in ('LIVE', 'UNSYNCED') and t_from <= t <= t_end:
            fr = frame_row(int(t.value), bids, asks, state == 'LIVE', S['degraded'] > 0, tick)
            if fr is not None:
                if state != 'LIVE': S['blocked'] += 1
                if fr['crossed']:
                    S['crossed'] += 1
                    if fr['healthy']: S['crossed_healthy'] += 1   # проскочили до детекции дыры
                F.append(fr)
            S['max_bid_lvl'] = max(S['max_bid_lvl'], len(bids))
            S['max_ask_lvl'] = max(S['max_ask_lvl'], len(asks))
        prev_t = t
    return dict(F=F, S=S, trans=trans, gap_ev=gap_ev, end_state=state)


def build(name, t_from, t_end, msgs=None, resync=None):
    path, sym = SRC[name]; tick = TICK[name]
    df, st = seed_book(path, sym, t_from)
    mid = (float(df.price.min()) + float(df.price.max())) / 2
    band = (mid * (1 - BAND_PCT), mid * (1 + BAND_PCT))
    if resync is None:
        resync = load_snapshots(path, sym, st, t_end)
    if msgs is None:
        msgs = iter_messages(path, sym, st, t_end, messages_for(path, sym, ()))
    return reconstruct(msgs, df, int(df['sequence_no'].iloc[0]), tick, GAP_THR[name],
                       t_from, t_end, resync=resync, band=band)


def frame_hash(F):
    d = pd.DataFrame(F)[['t_ns', 'pb', 'qb', 'pa', 'qa', 'healthy', 'degraded', 'crossed']]
    return hashlib.sha256(pd.util.hash_pandas_object(d, index=False).values.tobytes()).hexdigest()[:16]


print('Sequencer/Frame: reconstruct, build, frame_hash — определены; слои:', LAYERS)

Sequencer/Frame: reconstruct, build, frame_hash — определены; слои: Parser -> Sequencer -> Book -> Frame


## Пункт 1 — book builder по трём биржам

Прогоняем сборщик по окну. Дальше разбираем по отдельности всё, что бросилось в глаза: сбросов и
дубликатов нет ни у одной биржи, разрывы есть только у Bybit, и часть состояний помечена
непригодными. Флаг
`blocked` — это состояния после разрыва базового подфида: они выданы, но помечены `healthy=0`,
потому что до следующего снапшота потерянные обновления неотличимы от разреженного счётчика.
Тихо исключать их нельзя: кадр потерял бы непрерывность по времени. Поэтому при разрыве
базового подфида состояния помечаются `healthy=0`, но не выбрасываются из кадра, и потребитель
обязан фильтровать по этому флагу.

Что до перехода `UNSYNCED → RESYNC → SYNCING`: счётчик `resync` нулевой у всех трёх бирж, потому что
в окне ни одна биржа не получила снапшота, и recovery срабатывает только в инъекции
`insert_snapshot` (пункт 3). У Bybit на полном периоде следующие снапшоты — 21:10, 22:02, 22:50
и 22:58, но построчный прогон до них не доходит: окно кончается в 13:32. Это свойство данных,
а не баг сборщика.

In [8]:
BOOKS = {}
for name in SRC:
    t0 = time.time()
    BOOKS[name] = build(name, W0, W1)
    s = BOOKS[name]['S']
    print(f"{name:<8} сообщений={s['msgs']:>9} строк={s['rows']:>10} состояний={len(BOOKS[name]['F']):>8} "
          f"разрывов={s['gap']:>2} сбросов={s['reset']} дубликатов={s['dup_msg']} сирот={s['orphan']} "
          f"вне_полосы={s['invalid']:>8} resync={s['resyncs']} непригодных={s['blocked']:>7} "
          f"crossed={s['crossed']} уровни bid/ask max={s['max_bid_lvl']}/{s['max_ask_lvl']} "
          f"({time.time()-t0:.0f}s)")
    for e in BOOKS[name]['trans'][:4]:
        print('    ', e)

bybit    сообщений=   727415 строк=   4144599 состояний=  736793 разрывов=24 сбросов=0 дубликатов=0 сирот=1 вне_полосы=       0 resync=0 непригодных= 466026 crossed=0 уровни bid/ask max=51/51 (30s)
     ('2026-07-01 09:31:05.483193+03:00', 'SYNCING->LIVE', 1)
     ('2026-07-01 09:54:32.836297+03:00', 'GAP', 2)
     ('2026-07-01 09:58:10.916266+03:00', 'GAP', 2)
     ('2026-07-01 10:15:37.451381+03:00', 'GAP', 2)


okx      сообщений=  1212671 строк=  15252829 состояний= 1206867 разрывов= 0 сбросов=0 дубликатов=0 сирот=0 вне_полосы=       0 resync=0 непригодных=      0 crossed=0 уровни bid/ask max=401/401 (92s)
     ('2026-07-01 09:28:26.755265+03:00', 'SYNCING->LIVE', 1)


binance  сообщений=  5400109 строк=  22151130 состояний= 2232769 разрывов= 0 сбросов=0 дубликатов=0 сирот=2 вне_полосы= 2824537 resync=0 непригодных=      0 crossed=0 уровни bid/ask max=12892/11310 (165s)
     ('2026-07-01 09:29:21.103514+03:00', 'SYNCING->LIVE', 2)


## Гейт (a): книга совпадает с эталоном организаторов

Восстановленный стакан сверяем с эталоном `l2_executable_quotes_100ms`, где для каждой биржи
записаны `best_bid`, `best_ask` и размеры лучших уровней. Для каждой точки эталона берём
последнее состояние сборщика не позже неё (as-of соединение строго назад по receive-time,
`direction='backward'`) и сравниваем цены и размеры вплоть до точного совпадения.

Природу эталона проверяем до сверки, а не после: если бы он был построен из того же дампа тем же
алгоритмом, совпадение было бы самопроверкой и ничего не доказывало. Поэтому считаем долю
моментов эталона, которые совпадают с моментами сообщений соответствующего дампа, и смотрим на
поля, которых в L2-дампе нет. И разбираем, где именно лежали бы расхождения, — в разбивке по получасам.

In [9]:
ref = con.execute("""
    SELECT CAST(epoch(event_time) * 1000000 AS BIGINT) * 1000 AS t_ns, exchange,
           best_bid, best_ask, best_bid_amount, best_ask_amount, buy_vwap
    FROM read_parquet(?) WHERE asset='BTC' AND event_time>? AND event_time<=?
    ORDER BY event_time""", [REF, W0, W1]).df()
off_grid = float((np.abs(ref.buy_vwap * 10 - np.round(ref.buy_vwap * 10)) > 1e-6).mean())
print(f'доля buy_vwap эталона вне тик-сетки 0.1: {off_grid:.2%} — это средние цен по сделкам,')
print('а сделок в L2-дампе нет: часть полей эталона выведена из данных, которых у нас нет.')

PAIR = {'Bybit': 'bybit', 'Okex': 'okx', 'Binance': 'binance'}
match_rows = []
for ex, name in PAIR.items():
    path, sym = SRC[name]
    sub = con.execute("""WITH r AS (SELECT event_time FROM read_parquet(?) WHERE asset='BTC'
            AND exchange=? AND event_time>? AND event_time<=?),
          d AS (SELECT DISTINCT event_time FROM read_parquet(?) WHERE symbol=?
            AND event_time>? AND event_time<=?)
        SELECT count(*) AS n, count(d.event_time) AS m
        FROM r LEFT JOIN d USING (event_time)""",
        [REF, ex, W0, W1, path, sym, W0 - pd.Timedelta(minutes=2), W1 + pd.Timedelta(minutes=2)]).df().iloc[0]
    f = pd.DataFrame(BOOKS[name]['F'])
    st = pd.DataFrame({'t_ns': f.t_ns.astype('int64'),
                       'bid': f.pb * TICK[name], 'ask': f.pa * TICK[name],
                       'qb': f.qb, 'qa': f.qa})
    r = ref[ref.exchange == ex].copy().sort_values('t_ns')
    m = pd.merge_asof(r, st, on='t_ns', direction='backward')
    px = np.isclose(m.best_bid, m.bid, atol=1e-9) & np.isclose(m.best_ask, m.ask, atol=1e-9)
    qs = np.isclose(m.best_bid_amount, m.qb, atol=1e-9) & np.isclose(m.best_ask_amount, m.qa, atol=1e-9)
    match_rows.append(dict(venue=name, ref_points=len(r), time_subset=float(sub.m / sub.n),
                           price=float(px.mean()), size=float(qs.mean()),
                           both=float((px & qs).mean())))
    if name == 'binance':
        m['bad'] = ~(px & qs)
        m['ts'] = pd.to_datetime(m.t_ns, utc=True).dt.tz_convert('Europe/Moscow')
        by30 = m.groupby(m.ts.dt.floor('30min'))['bad'].agg(['mean', 'sum', 'count'])
print('совпадение с эталоном (точное: цены и размеры лучших уровней) и доля моментов эталона,')
print('лежащих точно на моментах сообщений фида:')
print(pd.DataFrame(match_rows).to_string(index=False, float_format=lambda v: f'{v:.4%}'))
print()
print('расхождения по получасам (нули — точное совпадение в каждом получасе):')
print(by30.to_string(float_format=lambda v: f'{v:.4f}'))

доля buy_vwap эталона вне тик-сетки 0.1: 2.73% — это средние цен по сделкам,
а сделок в L2-дампе нет: часть полей эталона выведена из данных, которых у нас нет.


совпадение с эталоном (точное: цены и размеры лучших уровней) и доля моментов эталона,
лежащих точно на моментах сообщений фида:
  venue  ref_points  time_subset     price      size      both
  bybit      138986    100.0000% 100.0000% 100.0000% 100.0000%
    okx      143005    100.0000% 100.0000% 100.0000% 100.0000%
binance      143935    100.0000% 100.0000% 100.0000% 100.0000%

расхождения по получасам (нули — точное совпадение в каждом получасе):
                            mean  sum  count
ts                                          
2026-07-01 09:30:00+03:00 0.0000    0  16796
2026-07-01 10:00:00+03:00 0.0000    0  17990
2026-07-01 10:30:00+03:00 0.0000    0  17992
2026-07-01 11:00:00+03:00 0.0000    0  17986
2026-07-01 11:30:00+03:00 0.0000    0  17996
2026-07-01 12:00:00+03:00 0.0000    0  17991
2026-07-01 12:30:00+03:00 0.0000    0  17991
2026-07-01 13:00:00+03:00 0.0000    0  17996
2026-07-01 13:30:00+03:00 0.0000    0   1197


Числа выше говорят об эталоне больше, чем о нашей книге. Все моменты времени эталона в
проверенном окне — ровно моменты сообщений соответствующего дампа (100 % с точностью до
микросекунды), его «100-мс сетка» — не ровная сетка, а время последнего сообщения в бакете, и цены с размерами лучших
уровней совпадают точно. Значит, эталон построен организаторами из тех же файлов, и
совпадение 100 % — это сверка нашей реконструкции с эталонной реализацией, а не независимое
подтверждение: общая для обеих реализаций систематическая ошибка (например, в порядке
применения сообщений) осталась бы невидимой. Сверка на ровной 100-мс сетке (так было устроено
в предыдущей домашке) давала ~89 % именно из-за несовпадения моментов сетки и сообщений;
после переноса эмиссии на границу сообщения as-of соединение попадает в то же сообщение, из
которого построен бакет эталона, поэтому 100 % здесь ожидаемы. Независимая часть эталона — поля `buy_vwap`/`sell_vwap`
(несколько процентов значений вне тик-сетки — средние цен сделок) и покрытие всего периода, а
не окна; против собственной ошибки нас защищают гейты (b) и (c): prefix-тест и счётчики
порядка не используют эталон вообще и падают на внутренних противоречиях реконструкции.

## Пункт 2 — набор инвариантов, который может провалиться

Инварианты оформлены как тесты со статусом pass/fail; у каждого есть условие провала, потому что
тест, который не может упасть, ничего не измеряет. Политика простая: брешь никогда не «чинят тихо».
Разрыв или сброс базового подфида — halt. Под halt здесь понимается не прекращение эмиссии
для биржи (как буквально звучит в слайде), а пометка всех состояний `healthy=0` до
восстановления: эмиссия продолжается, чтобы кадр не терял непрерывность по времени (иначе
строки пришлось бы тихо исключать), а потребитель фильтрует по флагу. Разрыв вторичного подфида — flag: top-of-book остаётся
пригодным, состояние помечается `degraded`. Невозможный размер обрезается и попадает в счётчик
(`size_cap`) — статус flag + count. У halt есть неизбежное запаздывание: дыра видна только по
первому сообщению после неё, поэтому состояния между началом дыры и детекцией выходят с
`healthy=1` — в fault injection это измеряется числом `crossed_healthy`.

Три группы: инварианты уровня сообщения (монотонность времени, отсутствие дубликатов, отсутствие
сброса нумерации, отсутствие невосстановленного разрыва, неотрицательная латентность), инварианты
уровня книги (best_bid < best_ask, размер в пределах капа, снятие только существующего уровня,
тик-сетка, предел глубины) и кросс-уровневые (эталон, повторный прогон, префикс, кросс-биржевая
согласованность) — последние три считаются в своих разделах.

In [10]:
def verdict(ok):
    return 'pass' if ok else 'FAIL'

msg_rows, book_rows = [], []
for name in SRC:
    s = BOOKS[name]['S']
    gap1 = sum(1 for e in BOOKS[name]['gap_ev'] if e[1] == 1); gap2 = s['gap'] - gap1
    msg_rows += [
        dict(venue=name, invariant='time_non_decreasing', breaches=s['time_back'],
             detail='убывание event_time вдоль порядка приёма', status=verdict(s['time_back'] == 0)),
        dict(venue=name, invariant='no_duplicate_message', breaches=s['dup_msg'],
             detail='повтор уже виденного sequence_no', status=verdict(s['dup_msg'] == 0)),
        dict(venue=name, invariant='no_sequence_reset', breaches=s['reset'],
             detail='шаг sequence_no назад в новое пространство', status=verdict(s['reset'] == 0)),
        dict(venue=name, invariant='no_unrecovered_gap', breaches=s['gap'],
             detail=f'шаг выше порога с паузой приёма (feed1: {gap1}, feed2: {gap2})',
             status=verdict(s['gap'] == 0)),
        dict(venue=name, invariant='latency_non_negative', breaches=s['latency_neg'],
             detail='receive-time раньше exchange-time', status=verdict(s['latency_neg'] == 0)),
    ]
    book_rows += [
        dict(venue=name, invariant='best_bid_lt_best_ask', breaches=s['crossed'],
             detail='эмитированное состояние crossed', status=verdict(s['crossed'] == 0)),
        dict(venue=name, invariant='size_within_cap', breaches=s['size_cap'],
             detail=f'размер уровня выше {MAX_SIZE:.0e} лотов', status=verdict(s['size_cap'] == 0)),
        dict(venue=name, invariant='delete_only_existing', breaches=s['orphan'],
             detail='снятие уровня глубже снапшотной базы (его add не публиковался); допустимый порог 10',
             status=verdict(s['orphan'] <= 10)),
    ]
IM = pd.DataFrame(msg_rows); IB = pd.DataFrame(book_rows)
print('инварианты уровня сообщения:')
print(IM.to_string(index=False))
print()
print('инварианты уровня книги:')
print(IB.to_string(index=False))
print()
print('независимая от эталона проверка логики set/delete: в чистом прогоне crossed=0 у всех')
print('трёх бирж; если бы amount=0 устанавливал размер 0 вместо удаления, best bid/ask остались')
print('бы на снятых ценах и книга регулярно пересекалась бы — слабый, но независимый аргумент.')
orph = ', '.join(f"{n}: {BOOKS[n]['S']['orphan']}" for n in SRC)
print(f'Сироты в чистом прогоне ({orph}) — все в первую секунду после снапшота: снятия feed1')
print('по уровням вне снапшотной базы, чей add не публиковался в нашем потоке; порог 10 —')
print('запас на порядок, массовые всплески (сотни, как в инъекциях) ловятся.')
print()
print('диагностика глубины (не гейт, а свойство данных):')
for name in SRC:
    s = BOOKS[name]['S']
    tag = 'OK' if s['max_ask_lvl'] <= SNAPDEPTH[name] * 1.5 else 'выше базы: уровни без снятий'
    df, st = seed_book(SRC[name][0], SRC[name][1], W0)
    lo, hi = float(df.price.min()), float(df.price.max())
    fr = pd.DataFrame(BOOKS[name]['F'])
    out_bid = float((fr.pb * TICK[name] < lo).mean()); out_ask = float((fr.pa * TICK[name] > hi).mean())
    print(f"  {name:<8} уровней bid/ask max = {s['max_bid_lvl']:>6}/{s['max_ask_lvl']:>6} "
          f"(снапшотная база {SNAPDEPTH[name]:>3}) — {tag}; лучший bid ниже дна базы в {out_bid:.1%} "
          f"состояний, лучший ask выше потолка — в {out_ask:.1%}")
print('  эталон содержит только лучший уровень: depth@N глубже best не верифицирован ни у одной')
print('  биржи, а уровни за пределами снапшотной базы не верифицированы (снятий для них нет).')

инварианты уровня сообщения:
  venue            invariant  breaches                                                detail status
  bybit  time_non_decreasing         0              убывание event_time вдоль порядка приёма   pass
  bybit no_duplicate_message         0                      повтор уже виденного sequence_no   pass
  bybit    no_sequence_reset         0            шаг sequence_no назад в новое пространство   pass
  bybit   no_unrecovered_gap        24 шаг выше порога с паузой приёма (feed1: 1, feed2: 23)   FAIL
  bybit latency_non_negative         0                     receive-time раньше exchange-time   pass
    okx  time_non_decreasing         0              убывание event_time вдоль порядка приёма   pass
    okx no_duplicate_message         0                      повтор уже виденного sequence_no   pass
    okx    no_sequence_reset         0            шаг sequence_no назад в новое пространство   pass
    okx   no_unrecovered_gap         0  шаг выше порога с паузой приёма

  bybit    уровней bid/ask max =     51/    51 (снапшотная база  50) — OK; лучший bid ниже дна базы в 58.6% состояний, лучший ask выше потолка — в 36.3%


  okx      уровней bid/ask max =    401/   401 (снапшотная база 400) — OK; лучший bid ниже дна базы в 33.8% состояний, лучший ask выше потолка — в 30.0%


  binance  уровней bid/ask max =  12892/ 11310 (снапшотная база 100) — выше базы: уровни без снятий; лучший bid ниже дна базы в 61.4% состояний, лучший ask выше потолка — в 31.4%
  эталон содержит только лучший уровень: depth@N глубже best не верифицирован ни у одной
  биржи, а уровни за пределами снапшотной базы не верифицированы (снятий для них нет).


## Гейт (b): отсутствие look-ahead — prefix-тест

Отсутствие look-ahead проверяется усечением входа: если строка на времени $t$ зависит только от
данных до $t$, то прогон на усечённом входе, остановленном в $t$, совпадёт с префиксом полного
прогона до $t$ включительно. Формально для любых $t_1 < t_2$:
$$\mathrm{Frame}\big(D_{[t_0, t_1]}\big) = \mathrm{Frame}\big(D_{[t_0, t_2]}\big)\big|_{t \le t_1}.$$
Берём 24 точки внутри получаса после первого снапшота (20 случайных с фиксированным seed и 4
якорные — 10 %, 25 %, 50 %, 75 %), каждый раз пересобираем книгу с нуля на усечённом входе и
сравниваем с префиксом полного прогона по кортежу `(t_ns, best_bid, size_bid, best_ask, size_ask)`.
Расхождение хотя бы в одной точке — провал гейта.

Оговорка про область: книга строится от снапшота, а снапшоты лежат только в начале файлов (у OKX и
Binance — по одному блоку, у Bybit — несколько в первый день). Поэтому точки усечения ставятся
внутри области, достижимой от снапшота без многодневного реплея; второй день покрыт агрегатными
инвариантами по полному периоду.

In [11]:
steps = sorted(set([0.10, 0.25, 0.50, 0.75] +
                        [round(float(x), 4) for x in np.random.default_rng(SEED).uniform(0.05, 0.95, 20)]))
KEYS = ('t_ns', 'pb', 'qb', 'pa', 'qa')
prefix_rows = []
for name in SRC:
    path, sym = SRC[name]; tick = TICK[name]
    df, st = seed_book(path, sym, W0)
    mid = (float(df.price.min()) + float(df.price.max())) / 2
    band = (mid * (1 - BAND_PCT), mid * (1 + BAND_PCT))
    P1 = W0 + pd.Timedelta(minutes=PREFIX_MIN)
    msgs = list(iter_messages(path, sym, st, P1, messages_for(path, sym, ())))
    def run_upto(cut):
        return reconstruct((m for m in msgs if m[0] <= cut), df, int(df['sequence_no'].iloc[0]),
                           tick, GAP_THR[name], W0, cut, resync=[], band=band)['F']
    full = run_upto(P1)
    lo, hi = full[0]['t_ns'], full[-1]['t_ns']
    mism = 0
    for x in steps:
        cutn = lo + int(x * (hi - lo))
        cut = pd.Timestamp(cutn, tz='UTC')
        tr = run_upto(cut)
        rf = [r for r in full if r['t_ns'] <= cutn]
        ok = len(tr) == len(rf) and all(all(tr[i][k] == rf[i][k] for k in KEYS) for i in range(len(rf)))
        mism += (not ok)
    prefix_rows.append(dict(venue=name, cuts=len(steps), mismatches=mism,
                            full_rows=len(full), status=verdict(mism == 0)))
print('prefix-тест (усечение входа против префикса полного прогона):')
print(pd.DataFrame(prefix_rows).to_string(index=False))

prefix-тест (усечение входа против префикса полного прогона):
  venue  cuts  mismatches  full_rows status
  bybit    24           0     101877   pass
    okx    24           0     160405   pass
binance    24           0     336777   pass


## Пункт 3 — fault injection

Инъекции делаются на копии данных с фиксированным seed. Берём пятиминутный кусок фида Bybit сразу
после снапшота (09:31:05) — до следующего снапшота далеко, поэтому книга не пересобирается и
инъекции не маскируются. Семь инъекций: пропуск сообщения, дубликат, перестановка соседних
сообщений, вставка снапшота в середину потока, порча размера лучшего уровня, сдвиг биржевых
часов (`exchange_time`) на 50 мс и сдвиг времени приёма (`event_time`) на 50 мс.

Два сдвига часов — случаи принципиально разного рода. `exchange_time` не участвует
ни в упорядочивании, ни в эмиссии, ни в джойнах — его сдвиг не может изменить книгу, и тест на
нём тривиален по построению: поймать его способен только счётчик латентности. Не смешиваем два
утверждения: «инвариант `latency_non_negative` нарушен» (1670 строк с отрицательной задержкой)
и «инъекция поймана» (pass) — второе верно именно потому, что счётчик среагировал; на
корректность книги эта инъекция не претендует. `event_time` — рабочий таймстемп пайплайна: его сдвиг обязан ломать
монотонность порядка приёма, хеш кадра и кросс-биржевой лаг. Заявлять поимку сдвигом
`exchange_time` через кросс-биржевой лаг было бы подтасовкой — этот сдвиг лаг не двигает.

Результат сравнивается с чистым прогоном того же куска, а не с абсолютным нулём: часть счётчиков
(например, сироты) не нуль и на живом фиде. Для каждой инъекции указано, какой инвариант её ловит,
какое поведение ожидается (resync, flag+count или флаг) и каким числом из прогона это
подтверждается — в таблице счётчики инвариантов, доля crossed-состояний под halt и совпадение
книги с эталоном на том же куске.

In [12]:
path, sym = SRC['bybit']
snap_df, SNAP_T = seed_book(path, sym, W0)
BT0, BT1 = SNAP_T, SNAP_T + pd.Timedelta(minutes=5)
msgs0 = list(iter_messages(path, sym, BT0, BT1, messages_for(path, sym, ())))
f1 = [i for i, (t, r) in enumerate(msgs0) if any(x[0] == 1 for x in r)]
mid = (float(snap_df.price.min()) + float(snap_df.price.max())) / 2
BAND = (mid * (1 - BAND_PCT), mid * (1 + BAND_PCT))
print('кусок:', BT0, '->', BT1, '| сообщений', len(msgs0), '| из них feed1', len(f1))


def copy_msgs(ms):
    return [(t, [tuple(x) for x in r]) for t, r in ms]


def inj_drop(ms):                       # теряем 500 сообщений feed1 -> номер уезжает вперёд
    drop = set(f1[300:800])
    return [m for i, m in enumerate(copy_msgs(ms)) if i not in drop]


def inj_duplicate(ms):                  # переотправляем старое сообщение -> номер назад, уже виден
    b = copy_msgs(ms)
    b.insert(f1[500], copy_msgs([ms[f1[200]]])[0])
    return b


def inj_swap(ms):                       # меняем местами соседние сообщения -> порядок времени нарушен
    b = copy_msgs(ms)
    i, j = f1[400], f1[401]
    b[i], b[j] = b[j], b[i]
    return b


def inj_corrupt_size(ms):               # портим размер крупнейшего уровня -> выше капа
    b = copy_msgs(ms); k = f1[600]
    t, r = b[k]
    tg = max(range(len(r)), key=lambda z: r[z][4])
    f, sq, sd, p, a, ex = r[tg]
    r[tg] = (f, sq, sd, p, MAX_SIZE * 10, ex)
    b[k] = (t, r)
    return b


def inj_clock_shift(ms):                # биржевое время +50 мс у 200 сообщений (ex — в наносекундах)
    b = copy_msgs(ms)
    for i in range(200, 400):
        t, r = b[i]
        b[i] = (t, [(f, sq, sd, p, a, ex + 5e7) for (f, sq, sd, p, a, ex) in r])
    return b


def inj_recv_shift(ms):                 # время приёма +50 мс у 200 сообщений -> ломается порядок
    d = pd.Timedelta(microseconds=50_000)
    b = copy_msgs(ms)
    for i in range(200, 400):
        t, r = b[i]
        b[i] = (t + d, r)
    return b


inj_snap = [(msgs0[400][0], snap_df.assign(sequence_no=10 ** 15))]   # снапшот в середину потока


def run_slice(ms, extra=()):
    return reconstruct(iter(ms), snap_df, int(snap_df['sequence_no'].iloc[0]), TICK['bybit'],
                       GAP_THR['bybit'], BT0, BT1, resync=list(extra),
                       feeds_seq=(1, 2), band=BAND)


cases = {'clean': (msgs0, ()),
         'drop_message': (inj_drop(msgs0), ()),
         'duplicate_message': (inj_duplicate(msgs0), ()),
         'swap_adjacent': (inj_swap(msgs0), ()),
         'corrupt_size': (inj_corrupt_size(msgs0), ()),
         'insert_snapshot': (copy_msgs(msgs0), inj_snap),
         'clock_shift_50ms': (inj_clock_shift(msgs0), ()),
         'receive_shift_50ms': (inj_recv_shift(msgs0), ())}

# совпадение книги на этом же куске с эталоном — потребительская метрика качества после инъекции
ref_slice = con.execute("""SELECT CAST(epoch(event_time)*1000000 AS BIGINT)*1000 AS t_ns,
    best_bid, best_ask FROM read_parquet(?) WHERE asset='BTC' AND exchange='Bybit'
    AND event_time>? AND event_time<=? ORDER BY event_time""", [REF, BT0, BT1]).df()

fault_rows = []
for lab, (ms, ex) in cases.items():
    o = run_slice(ms, ex); s = o['S']
    fr = pd.DataFrame(o['F']).sort_values('t_ns')
    mm = pd.merge_asof(ref_slice, pd.DataFrame({'t_ns': fr.t_ns.astype('int64'),
                                                 'bid': fr.pb * TICK['bybit'],
                                                 'ask': fr.pa * TICK['bybit']}),
                       on='t_ns', direction='backward')
    rm = float((np.isclose(mm.best_bid, mm.bid, atol=1e-9)
                & np.isclose(mm.best_ask, mm.ask, atol=1e-9)).mean())
    fault_rows.append(dict(case=lab, gap=s['gap'], reset=s['reset'], dup=s['dup_msg'],
                           orphan=s['orphan'], size_cap=s['size_cap'], time_back=s['time_back'],
                           lat_neg=s['latency_neg'], crossed=s['crossed'],
                           cr_healthy=s['crossed_healthy'], blocked=s['blocked'],
                           resync=s['resyncs'], ref_match=rm, hash=frame_hash(o['F'])))
FT = pd.DataFrame(fault_rows).set_index('case')
print(FT.to_string())

кусок: 2026-07-01 09:31:05.480956+03:00 -> 2026-07-01 09:36:05.480956+03:00 | сообщений 17673 | из них feed1 10882


                    gap  reset  dup  orphan  size_cap  time_back  lat_neg  crossed  cr_healthy  blocked  resync  ref_match              hash
case                                                                                                                                        
clean                 0      0    0       1         0          0        0        0           0        0       0   1.000000  490d122be3d3701b
drop_message          1      0    0      62         0          0        0     6579         176    16466       0   0.553358  6550dc1ba13e443d
duplicate_message     1      0    1       3         0          1        0      734           1    16889       0   0.965564  a4ee2c9d8f3696d2
swap_adjacent         0      1    0       1         0          2        0        0           0    17042       0   1.000000  4770625c44d5ec5e
corrupt_size          0      0    0       1         1          0        0        0           0        0       0   1.000000  490d122be3d3701b
insert_snapsh

In [13]:
clean = FT.loc['clean']
expected = {
    'drop_message':      ('no_unrecovered_gap', 'halt: состояния после детекции выходят с healthy=0'),
    'duplicate_message': ('no_duplicate_message', 'flag+count: дубликат отброшен по seen-set'),
    'swap_adjacent':     ('time_non_decreasing', 'flag: порядок времени нарушен, далее reset'),
    'corrupt_size':      ('size_within_cap', 'flag+count: размер обрезан и посчитан'),
    'insert_snapshot':   ('resync', 'resync: книга заменяется, replay hash расходится с чистым'),
    'clock_shift_50ms':  ('latency_non_negative', 'flag+count (тривиально): только реакция счётчика латентности'),
    'receive_shift_50ms': ('time_non_decreasing', 'flag: сдвиг event_time ломает порядок и меняет hash'),
}
catch = []
for lab in FT.index:
    if lab == 'clean':
        continue
    inv, beh = expected[lab]
    if lab == 'clock_shift_50ms':
        caught = FT.loc[lab, 'lat_neg'] > clean['lat_neg']
        ev = (f"тест тривиален: exchange_time не входит ни в порядок, ни в эмиссию, книга "
              f"не меняется (match {FT.loc[lab,'ref_match']:.2%}); проверяется только реакция "
              f"счётчика латентности: lat_neg {int(FT.loc[lab,'lat_neg'])} (clean {int(clean['lat_neg'])})")
    elif lab == 'receive_shift_50ms':
        caught = (FT.loc[lab, 'time_back'] > clean['time_back']
                  and FT.loc[lab, 'hash'] != clean['hash'])
        ev = (f"time_back {int(FT.loc[lab,'time_back'])} (clean {int(clean['time_back'])}); "
              f"hash {clean['hash']} -> {FT.loc[lab,'hash']}")
    elif lab == 'insert_snapshot':
        caught = FT.loc[lab, 'resync'] > 0 and FT.loc[lab, 'hash'] != clean['hash']
        ev = (f"resync {int(FT.loc[lab,'resync'])}; hash {clean['hash']} -> {FT.loc[lab,'hash']}; "
              f"match {FT.loc[lab,'ref_match']:.2%} (clean {clean['ref_match']:.2%})")
    elif lab == 'corrupt_size':
        caught = FT.loc[lab, 'size_cap'] > clean['size_cap']
        ev = (f"size_cap {int(FT.loc[lab,'size_cap'])} (clean {int(clean['size_cap'])}); "
              f"цена best не задета: match {FT.loc[lab,'ref_match']:.2%}")
    elif lab == 'drop_message':
        caught = FT.loc[lab, 'gap'] > clean['gap']
        ev = (f"gap {int(FT.loc[lab,'gap'])} (clean {int(clean['gap'])}); crossed "
              f"{int(FT.loc[lab,'crossed'])}, из них под halt "
              f"{int(FT.loc[lab,'crossed'] - FT.loc[lab,'cr_healthy'])}, до детекции с healthy=1 "
              f"{int(FT.loc[lab,'cr_healthy'])} (clean 0); blocked {int(FT.loc[lab,'blocked'])}; "
              f"match {FT.loc[lab,'ref_match']:.2%} (clean {clean['ref_match']:.2%})")
    elif lab == 'duplicate_message':
        caught = FT.loc[lab, 'dup'] > clean['dup']
        ev = (f"dup {int(FT.loc[lab,'dup'])} (clean {int(clean['dup'])}); "
              f"match {FT.loc[lab,'ref_match']:.2%} (clean {clean['ref_match']:.2%})")
    else:
        caught = FT.loc[lab, 'time_back'] > clean['time_back'] or FT.loc[lab, 'reset'] > clean['reset']
        ev = f"time_back {int(FT.loc[lab,'time_back'])}, reset {int(FT.loc[lab,'reset'])} (clean 0)"
    catch.append(dict(fault=lab, invariant_shown=inv, caught=verdict(caught),
                      expected_behaviour=beh, evidence=ev))
CT = pd.DataFrame(catch)
print('fault -> invariant that caught it -> behaviour:')
print(CT.to_string(index=False))
print()
print('каждая инъекция поймана:', bool(CT['caught'].eq('pass').all()),
      '| чистый прогон без нарушений:',
      bool(FT.loc['clean', ['gap', 'reset', 'dup', 'size_cap', 'time_back',
                           'lat_neg', 'cr_healthy']].abs().sum() == 0))

fault -> invariant that caught it -> behaviour:
             fault      invariant_shown caught                                           expected_behaviour                                                                                                                                                                       evidence
      drop_message   no_unrecovered_gap   pass           halt: состояния после детекции выходят с healthy=0                                        gap 1 (clean 0); crossed 6579, из них под halt 6403, до детекции с healthy=1 176 (clean 0); blocked 16466; match 55.34% (clean 100.00%)
 duplicate_message no_duplicate_message   pass                    flag+count: дубликат отброшен по seen-set                                                                                                                                  dup 1 (clean 0); match 96.56% (clean 100.00%)
     swap_adjacent  time_non_decreasing   pass                   flag: порядок времени нарушен, далее r

## Гейт (d): воспроизводимость

Воспроизводимость подтверждаем тремя способами. Первый — фиксируем хеши входных файлов:
подмена файла будет видна. Второй — прогоняем сборщик дважды на одном входе, и хеш состояния
обязан совпасть. Третий — приводим команду
пересборки: ноутбук запускается как
`jupyter nbconvert --to notebook --execute --inplace hw5.ipynb`, а все параметры (окно, число точек
усечения, seed) заданы в первой ячейке, поэтому пересборка не требует ручных правок. От числа потоков ничего не зависит: вход заранее упорядочен `ORDER BY event_time, file_row_number`, и результат
зависит только от этого порядка.

In [14]:
def sha256_file(p, chunk=1 << 24):
    h = hashlib.sha256()
    with open(p, 'rb') as fh:
        for b in iter(lambda: fh.read(chunk), b''):
            h.update(b)
    return h.hexdigest()


in_hashes = {os.path.basename(p): sha256_file(p)[:16] for p, _ in SRC.values()}
in_hashes[os.path.basename(REF)] = sha256_file(REF)[:16]
print('входные хеши (sha256, первые 16 знаков):')
for k, v in in_hashes.items():
    print(f'  {k:<48} {v}')

run_a = {name: frame_hash(build(name, W0, W1)['F']) for name in SRC}
run_b = {name: frame_hash(build(name, W0, W1)['F']) for name in SRC}
print()
print('выходные хеши состояния на двух независимых прогонах:')
for name in SRC:
    print(f'  {name:<8} {run_a[name]}  {run_b[name]}  {verdict(run_a[name] == run_b[name])}')
print()
print('команда пересборки: jupyter nbconvert --to notebook --execute --inplace hw5.ipynb')

входные хеши (sha256, первые 16 знаков):
  bybit_l2_20260701_20260703.parquet               a50ed47bab5a5c31
  okex_l2_20260701_20260703.parquet                f847e7f0f410ad59
  binance_l2_20260701_20260703.parquet             de867e8be19d93b5
  l2_executable_quotes_100ms_10000.parquet         759def93f974c66d



выходные хеши состояния на двух независимых прогонах:
  bybit    a659336e744015bc  a659336e744015bc  pass
  okx      acc2e2dda05515ea  acc2e2dda05515ea  pass
  binance  3d308c7a202ab81d  3d308c7a202ab81d  pass

команда пересборки: jupyter nbconvert --to notebook --execute --inplace hw5.ipynb


## Пункт 4 — feature frame: событийный и 100-мс сетка

Event-time frame — одна строка на сообщение, эмитится строго point-in-time (гейт b). На каждой
строке: `spread_bps`, глубина по сторонам, queue imbalance, microprice, OFI и `age_ms`. Спред в
базисных пунктах — $s = (a - b)/\mu \cdot 10^4$, где $\mu = (a+b)/2$; глубина на $N$ уровнях —
сумма размеров лучших $N$ уровней стороны; queue imbalance по лучшим уровням —
$\mathrm{OBI} = (V_{bid} - V_{ask})/(V_{bid} + V_{ask})$, где $V$ — размеры лучших уровней; microprice —
$$\mathrm{mp} = \frac{P_{ask}\, V_{bid} + P_{bid}\, V_{ask}}{V_{bid} + V_{ask}};$$
`age_ms` — сколько миллисекунд прошло с предыдущего сообщения фида, то есть насколько строка
устарела к моменту публикации; считается по любому сообщению фида, независимо от того,
касалось ли оно лучшего уровня.

OFI считается по формуле Конту-Куканова-Стойкова:
$$e = \mathbb{1}[P_b \ge P_b']\,q_b - \mathbb{1}[P_b \le P_b']\,q_b'
      - \mathbb{1}[P_a \le P_a']\,q_a + \mathbb{1}[P_a \ge P_a']\,q_a',$$
где штрихом помечено предыдущее состояние. Реализация через абсолютные размеры с индикаторами
эквивалентна определению через прирост очереди $\Delta q$: при неизменной цене индикаторы дают
разность $q - q'$, а при новом уровне — новый размер или старый, то есть ровно $\Delta q$. С L2 нельзя подписать поток сделок: tape в выгрузке
отсутствует. Поэтому OFI заменяет trade-signed flow: он измеряет изменение очереди на лучших
уровнях, а не агрессию в сделках.

Глубина `depth_bid`/`depth_ask` показана на OKX: его снапшотная база 400 уровней покрывает
top-10 с запасом; у Binance база 100, а уровни за ней идут без снятий, поэтому depth@10 у
Binance в анализ не берётся (см. пункт 5).

Сетка 100 мс — производное представление, а не второй источник данных. Она привязана к первому
событию окна (не к границе эпохи) и берётся последним наблюдением вперёд (LOCF), но вместе с
`age_ms`, чтобы staleness была видна: строка сетки без `age_ms` создавала бы иллюзию, что
котировка обновилась ровно в этот такт. `age_ms` на сетке считается от времени последнего
реального сообщения, присоединённого backward, а не от времени предыдущего такта.

In [15]:
FEAT_VENUE = 'okx'      # event-time frame показываем на OKX: снапшотная база 400 покрывает top-10
f = pd.DataFrame(BOOKS[FEAT_VENUE]['F'])
f['ts'] = pd.to_datetime(f.t_ns, utc=True).dt.tz_convert('Europe/Moscow')
f['mid'] = (f.pb + f.pa) / 2 * TICK[FEAT_VENUE]
f['ofi'] = np.nan

pb, qb = f.pb.to_numpy(), f.qb.to_numpy()
pa, qa = f.pa.to_numpy(), f.qa.to_numpy()
e = (np.where(pb[1:] >= pb[:-1], qb[1:], 0) - np.where(pb[1:] <= pb[:-1], qb[:-1], 0)
     - np.where(pa[1:] <= pa[:-1], qa[1:], 0) + np.where(pa[1:] >= pa[:-1], qa[:-1], 0))
f.loc[f.index[1:], 'ofi'] = e
f['age_ms'] = f.t_ns.diff() / 1e6


# worked example из задания: bid 100.00 x10 -> x14 ; ask 100.01 x12 -> x9
def ofi_worked(Pb0, qb0, Pb1, qb1, Pa0, qa0, Pa1, qa1):
    return ((qb1 if Pb1 >= Pb0 else 0) - (qb0 if Pb1 <= Pb0 else 0)
            - (qa1 if Pa1 <= Pa0 else 0) + (qa0 if Pa1 >= Pa0 else 0))


print('OFI worked example: bid 100.00 x10 -> x14, ask 100.01 x12 -> x9,',
      'e =', ofi_worked(10000, 10, 10000, 14, 10001, 12, 10001, 9), '(ожидание +7)')
print()
print(f'event-time frame {FEAT_VENUE}: строк {len(f)}')
cols = ['ts', 'pb', 'qb', 'pa', 'qa', 'spread_bps', 'depth_bid', 'depth_ask', 'obi', 'micro', 'ofi', 'age_ms']
print(f[cols].head(8).to_string(index=False, float_format=lambda v: f'{v:.4f}'))
print()
print('описательные статистики event-time frame:')
print(f[['spread_bps', 'depth_bid', 'depth_ask', 'obi', 'age_ms']].describe()
      .to_string(float_format=lambda v: f'{v:.4f}'))

# derived 100 ms grid view (LOCF + age_ms)
g = pd.DataFrame({'t_ns': np.arange(int(f.t_ns.iloc[0]), int(f.t_ns.iloc[-1]) + 1, 100_000_000)})
grid = pd.merge_asof(g, f[['t_ns', 'pb', 'qb', 'pa', 'qa', 'micro', 'depth_bid', 'depth_ask', 'obi']],
                     on='t_ns', direction='backward')
upd = pd.Index(f.t_ns.to_numpy())
grid['age_ms'] = (grid['t_ns'].to_numpy() - upd.to_series().reindex(pd.Index(grid['t_ns']),
                                                                    method='ffill').to_numpy()) / 1e6
grid['mid'] = (grid.pb + grid.pa) / 2 * TICK[FEAT_VENUE]
grid['spread_bps'] = (grid.pa - grid.pb) * TICK[FEAT_VENUE] / grid.mid * 1e4
print()
print(f'сетка 100 мс: строк {len(grid)} (event-time строк {len(f)}, отношение {len(grid)/len(f):.3f})')
print('age_ms на сетке, мс: p50 %.1f  p95 %.1f  p99 %.1f  max %.1f'
      % tuple(grid.age_ms.quantile(q) for q in (0.5, 0.95, 0.99, 1.0)))
print(grid[['t_ns', 'pb', 'pa', 'mid', 'spread_bps', 'age_ms']].head(5)
      .to_string(index=False, float_format=lambda v: f'{v:.4f}'))

OFI worked example: bid 100.00 x10 -> x14, ask 100.01 x12 -> x9, e = 7 (ожидание +7)

event-time frame okx: строк 1206867
                              ts     pb     qb     pa     qa  spread_bps  depth_bid  depth_ask    obi       micro     ofi  age_ms
2026-07-01 09:32:00.018052+03:00 587795 4.2926 587796 0.1839      0.0170     5.1561     2.0725 0.9178 587795.9589     NaN     NaN
2026-07-01 09:32:00.018265+03:00 587795 4.2926 587796 0.1839      0.0170     5.1561     2.0725 0.9178 587795.9589  0.0000  0.2130
2026-07-01 09:32:00.028778+03:00 587795 4.2926 587796 0.1839      0.0170     5.1561     2.0725 0.9178 587795.9589  0.0000 10.5130
2026-07-01 09:32:00.038289+03:00 587795 4.2926 587796 0.1759      0.0170     5.1561     2.0645 0.9213 587795.9606  0.0080  9.5110
2026-07-01 09:32:00.039262+03:00 587795 4.2926 587796 0.1759      0.0170     5.1561     2.0645 0.9213 587795.9606  0.0000  0.9730
2026-07-01 09:32:00.048135+03:00 587795 4.2926 587796 0.1760      0.0170     5.1561     2.0646 0.9

In [16]:
# microprice против mid и информативность OFI (только точка-в-точке, без заглядывания вперёд)
ok = f[['micro', 'mid', 'obi']].notna().all(axis=1)
ff = f[ok].copy()
ff['mid_tick'] = (ff.pb + ff.pa) / 2          # mid в тех же тиках, что и micro
print('microprice != mid:', f'{float((ff.micro != ff.mid_tick).mean()):.2%}',
      '| среднее |micro-mid|, тиков:', f'{float((ff.micro - ff.mid_tick).abs().mean()):.4f}')
# dmid_next — будущее изменение mid: это target для оценки информативности,
# в feature frame не входит; фичи (ofi/obi) берутся в момент t, look-ahead в фичах нет
ff['dmid_next'] = ff.mid.shift(-1) - ff.mid
sub = ff[['ofi', 'obi', 'dmid_next']].dropna()
print('corr(OFI_t, dmid_{t+1}) =', f'{float(sub.ofi.corr(sub.dmid_next)):.4f}',
      '| corr(OBI_t, dmid_{t+1}) =', f'{float(sub.obi.corr(sub.dmid_next)):.4f}',
      f'| n = {len(sub)}')
print()
print('Границы данных: effective и realised spread невычислимы (нет сделок), queue position')
print('невычислима (нет позиции в очереди и cancel/replace с id заявок). OFI — прокси для')
print('подписанного потока сделок, которого в L2 нет.')

microprice != mid: 100.00% | среднее |micro-mid|, тиков: 0.4501
corr(OFI_t, dmid_{t+1}) = 0.0395 | corr(OBI_t, dmid_{t+1}) = 0.1477 | n = 1206865

Границы данных: effective и realised spread невычислимы (нет сделок), queue position
невычислима (нет позиции в очереди и cancel/replace с id заявок). OFI — прокси для
подписанного потока сделок, которого в L2 нет.


## Кросс-биржевая часть: консолидированный BBO и дислокации

Джойны между биржами делаются исключительно по времени приёма `event_time` (биржевые штампы
разъезжаются, а наше время приёма общее для трёх потоков). На 100-мс такте берём по каждой бирже
последнюю котировку не позже такта (as-of назад) и строим консолидированный BBO: лучший bid и
лучший ask среди бирж с атрибуцией площадки и торгуемым размером $\min(V_A, V_B)$.

Дислокация считается как
$$g = \frac{P^{bid}_B - P^{ask}_A}{P^{ask}_A} \cdot 10^4, \qquad g - f_A - f_B > 0,$$
где $f_A, f_B$ — taker-комиссии. Это флаг расхождения котировок, а не P&L: не учтены доступный
размер, латентность и влияние собственной заявки. Комиссии — заявленное допущение: base-tier taker,
Bybit 0.055 %, OKX и Binance 0.050 % (публичные тарифные таблицы бирж; страница Bybit Trading Fee
Structure обновлена 2026-09-02, тарифы OKX и Binance сверены 2026-10-04). Реальные ставки зависят
от VIP-уровня и региона, поэтому показана и зависимость числа флагов от ставки комиссии.

Отдельно проверяется сдвиг часов приёма как fault-инъекция: пересобираем книгу Bybit на
тридцатиминутном окне с `event_time`, сдвинутым на +50 мс, и смотрим, куда уходит максимум
кросс-корреляции приростов mid против OKX на 100-мс такте. Сдвиг должен быть виден по числам: без
него максимум корреляции стоит при нулевом лаге, после сдвига — при лаге в один такт. Сдвигаем вход, а не уже собранный ряд: `shift(1)` по готовой сетке
показал бы арифметику сдвига, а не поведение пайплайна, поэтому книга пересобирается целиком. Оговорка к этой метрике:
50 мс — полтакта 100-мс сетки, поэтому по лагу сдвиг неотличим от естественного отставания
площадки: корреляция меняется в третьем знаке (0.624 → 0.636), а лаг уходит на соседний такт.
Это ограничение разрешения сетки, а не доказательство поимки; поимка подтверждается
`time_back=1` и изменением хеша кадра в таблице fault injection.

In [17]:
FEE_TAKER = {'bybit': 5.5e-4, 'okx': 5.0e-4, 'binance': 5.0e-4}
STEP = 100_000_000     # 100 мс

G = None
for name in SRC:
    d = pd.DataFrame(BOOKS[name]['F'])[['t_ns', 'pb', 'pa', 'qb', 'qa']].copy()
    d['g'] = (d.t_ns // STEP) * STEP
    d = d.groupby('g').last()[['pb', 'pa', 'qb', 'qa']]
    d = d.rename(columns={c: f'{c}_{name}' for c in ('pb', 'pa', 'qb', 'qa')})
    G = d if G is None else G.join(d, how='outer')
G = G.sort_index().ffill()
print('тактов консолидации:', len(G))

rows = []
for ts, r in G.iterrows():
    q = {n: (r[f'pb_{n}'], r[f'pa_{n}'], r[f'qb_{n}'], r[f'qa_{n}']) for n in SRC}
    for A in SRC:
        for Bv in SRC:
            if A == Bv:
                continue
            pbA, paA, pbB = q[A][0], q[A][1], q[Bv][0]
            if not (np.isfinite(pbA) and np.isfinite(paA) and np.isfinite(pbB)) or paA <= 0:
                continue
            gross = (pbB - paA) / paA * 1e4
            rows.append(dict(t=ts, buy_at=A, sell_at=Bv, gross_bps=gross,
                             net_bps=gross - (FEE_TAKER[A] + FEE_TAKER[Bv]) * 1e4,
                             tradable=min(q[A][3], q[Bv][2])))
DIS = pd.DataFrame(rows)
print('пар наблюдений', len(DIS),
      '| gross>0:', f'{float((DIS.gross_bps > 0).mean()):.4%}',
      '| net>0:', f'{float((DIS.net_bps > 0).mean()):.4%}',
      f'({int((DIS.net_bps > 0).sum())} наблюдений)')
if (DIS.net_bps > 0).any():
    print('медиана торгуемого размера на флагах, контрактов:',
          round(float(DIS.loc[DIS.net_bps > 0, 'tradable'].median()), 3))
print()
print('топ-3 по net (дислокация котировок, не P&L):')
print(DIS.nlargest(3, 'net_bps').to_string(index=False, float_format=lambda v: f'{v:.3f}'))
print()
print('зависимость числа флагов от ставки комиссии (флаг, если gross > f_A + f_B):')
for fv in (0.01, 0.02, 0.05, 0.10):
    n = int((DIS.gross_bps > 2 * fv * 100).sum())
    print(f'  одинаковая ставка {fv:.2f}% на обе стороны -> флагов {n} ({n/len(DIS):.5%})')

# сдвиг часов приёма: пересборка книги Bybit на 30-мин окне с event_time +50 мс
def lag_scan(a, b, lags=range(-6, 7)):
    out = []
    for L in lags:
        x = a[L:] if L >= 0 else a[:L]
        y = b[:len(b) - L] if L >= 0 else b[-L:]
        n = min(len(x), len(y))
        out.append((L, float(np.corrcoef(x[:n], y[:n])[0, 1])))
    return max(out, key=lambda z: z[1])


def slice_frame(name, minutes, shift_ns=0):
    # книга на срезе окна; shift_ns сдвигает receive-time всего среза (часы гейта отстают)
    p, s = SRC[name]; tick = TICK[name]
    df, st = seed_book(p, s, W0)
    m = (float(df.price.min()) + float(df.price.max())) / 2
    band = (m * (1 - BAND_PCT), m * (1 + BAND_PCT))
    t1 = W0 + pd.Timedelta(minutes=minutes)
    ms = iter_messages(p, s, st, t1, messages_for(p, s, ()))
    if shift_ns:
        d = pd.Timedelta(microseconds=shift_ns // 1000)
        ms = ((t + d, r) for t, r in ms)
    o = reconstruct(ms, df, int(df['sequence_no'].iloc[0]), tick, GAP_THR[name], W0, t1,
                    resync=[], band=band)
    return pd.DataFrame(o['F'])[['t_ns', 'pb', 'pa']].sort_values('t_ns')


def best_lag(bby, oky):
    lo = max(int(bby.t_ns.min()), int(oky.t_ns.min())); hi = min(int(bby.t_ns.max()), int(oky.t_ns.max()))
    GT = pd.DataFrame({'g': np.arange(lo, hi + 1, STEP)})
    bb = pd.merge_asof(GT, bby, left_on='g', right_on='t_ns', direction='backward')
    ok = pd.merge_asof(GT, oky, left_on='g', right_on='t_ns', direction='backward')
    dm = pd.DataFrame({'o': (ok.pb + ok.pa) / 2, 'b': (bb.pb + bb.pa) / 2}).diff().dropna()
    return lag_scan(dm.o.to_numpy(), dm.b.to_numpy())


bb_c = slice_frame('bybit', 30)
ok_c = slice_frame('okx', 30)
bb_s = slice_frame('bybit', 30, shift_ns=STEP // 2)      # +50 мс = полтакта сетки
lag0, c0 = best_lag(bb_c, ok_c)
lag1, c1 = best_lag(bb_s, ok_c)
print()
print(f'кросс-корреляция приростов mid OKX vs Bybit (30-мин окно): '
      f'без сдвига лучший лаг {lag0} тактов (corr {c0:.3f}); после сдвига Bybit на 50 мс — '
      f'лаг {lag1} тактов (corr {c1:.3f})')

тактов консолидации: 144000


пар наблюдений 864000 | gross>0: 48.6332% | net>0: 0.0000% (0 наблюдений)

топ-3 по net (дислокация котировок, не P&L):
                  t  buy_at sell_at  gross_bps  net_bps  tradable
1782887824600000000 binance   bybit      8.695   -1.805     0.250
1782887672700000000 binance   bybit      7.683   -2.817     1.525
1782887672700000000     okx   bybit      7.206   -3.294     1.525

зависимость числа флагов от ставки комиссии (флаг, если gross > f_A + f_B):
  одинаковая ставка 0.01% на обе стороны -> флагов 1376 (0.15926%)
  одинаковая ставка 0.02% на обе стороны -> флагов 33 (0.00382%)
  одинаковая ставка 0.05% на обе стороны -> флагов 0 (0.00000%)
  одинаковая ставка 0.10% на обе стороны -> флагов 0 (0.00000%)



кросс-корреляция приростов mid OKX vs Bybit (30-мин окно): без сдвига лучший лаг 0 тактов (corr 0.624); после сдвига Bybit на 50 мс — лаг -1 тактов (corr 0.636)


## Пункт 5 — отчёт: здоровье площадок, находки, допущения, границы данных

**Здоровье площадок.** В четырёхчасовом окне Bybit дал 24 разрыва и ни одного сброса: 23 из них
во вторичном `feed2` (top-of-book не задет, состояния помечены `degraded`) и один в базовом
`feed1` на 10:55:57 — после него книга уходит в halt, и 466 026 состояний (63 % кадра) выходят
с `healthy=0` до конца окна, потому что следующий снапшот только в 21:10. У OKX и Binance в окне разрывов
и сбросов нет. Снапшотов в окне тоже нет ни у одной биржи, поэтому переход
`UNSYNCED → RESYNC → SYNCING` на живых данных не сработал ни разу (`resync=0`) и продемонстрирован
только инъекцией `insert_snapshot`; на полном периоде следующие снапшоты Bybit — 21:10 и позже,
но построчный прогон до них не доходит — это свойство данных, а не баг (см. пункт 1). По полному периоду разрывы почти исключительно у Bybit (336,
из них 328 в `feed2` и 8 в `feed1`), у OKX и Binance — по одному-два в час самого начала
файла, что похоже на переходный процесс после снапшота, а не на потерю данных; сбросов и
дубликатов нет ни у одной биржи. Медианная задержка приёма
различается в разы: Binance около 2 мс, OKX около 24 мс, Bybit около 37 мс; отрицательных задержек
нет, но хвосты доходят до 288 мс (OKX; у Binance максимум 164 мс). Сравнивать площадки по абсолютной задержке как «быстрее/медленнее»
можно лишь с оговоркой: в число входят и путь доставки, и география гейта.

**Находки.** Первое — порядок эмиссии. Если публиковать состояние в начале сообщения, а не в конце,
строка на времени $t$ зависит от первой строки следующего сообщения, то есть появляется look-ahead
в один апдейт. После переноса эмиссии на границу сообщения совпадение с эталоном стало точным,
а prefix-тест перестал находить расхождения на границах усечения. Точное совпадение при этом
означает ровно то, что написано в гейте (a): эталон выведен из того же дампа, и его бакет
закрывается тем же сообщением, на котором эмитируем мы. Природа такой сверки разобрана там же;
независимую часть проверки несут prefix-тест и счётчики порядка.

Второе — все подфиды. У Binance поток снятий идёт отдельным подфидом 4; если собирать только 1/2/3,
в top-of-book появляются ложные crossed-состояния (3.5 % в первые два часа), а с подфидом 4 их
остаётся ноль. Это видно в диагностике: `crossed` равно нулю у всех трёх бирж только при полном
наборе подфидов, а совпадение цен и размеров лучших уровней с эталоном — 100.0000 % (с оговоркой
о природе эталона из гейта (a)). Есть и независимая от эталона проверка логики
set/delete: в чистом прогоне `crossed=0` у всех трёх бирж; если бы `amount=0` устанавливал
размер 0 вместо удаления, best bid/ask оставались бы на снятых ценах и книга регулярно
пересекалась бы — это слабый, но независимый аргумент.

Третье — глубина Binance. В восстановленном стакане Binance глубина доходит до 12 892 bid- и
11 310 ask-уровней (медиана около 10 тысяч) при снапшотной базе 100 — на два порядка выше. Причина в том, что инкременты несут новые цены за пределами
снапшотной глубины, а снятий для них в дампе нет. Top-of-book от этого не страдает: лучший уровень и его размер верифицированы эталоном
точно, а book-инварианты (сортировка, уникальность уровня, кап размера) держатся —
SortedDict исключает дубликаты по построению, а `size_within_cap` проходит. Но лучший bid
Binance уходит ниже дна снапшотной базы в 61 % состояний, а лучший ask — выше потолка в 31 %,
поэтому depth@10 у Binance часто считается уже за пределами верифицируемой базы, и глубина
глубже best-уровня не верифицирована ни у одной биржи (эталон содержит только top). Это
ограничение вынесено в границы данных.

**Допущения.** Тик 0.1 и лот (Bybit и Binance 0.001, OKX 0.0001) выведены из данных и сверены
с конфигурацией — расхождений нет. Комиссии — base-tier taker из
публичных тарифов (Bybit 0.055 %, OKX/Binance 0.050 %), сверены 2026-10-04; VIP-ставки ниже. Что
именно биржа называет `exchange_time` (момент матчинга или отправки), из данных не видно, поэтому
задержка трактуется как верхняя оценка. Окно реконструкции — 4 часа после первого снапшота; по
полному периоду считаются агрегаты, а не построчная книга.

**Границы данных.** С L2 нельзя подписать поток сделок — tape отсутствует, поэтому OFI заменяет
trade-signed flow. Effective и realised spread невычислимы (нет сделок), queue position
невычислима (нет позиции в очереди). Кросс-биржевые флаги дислокации — не P&L: не учтены размер,
латентность и влияние собственной заявки. Все кросс-биржевые сопоставления сделаны по времени
приёма, поэтому ошибка хронометража гейта переносится на них целиком. Глубина Binance за
снапшотной базой недостоверна, а depth@N глубже лучшего уровня не верифицирован ни у одной
биржи; сверка с эталоном — не независимое подтверждение (см. гейт (a)).

In [18]:
print('СВОДКА ГЕЙТОВ')
print()
print('(a) correct book  : match цен и размеров с эталоном ' +
      ', '.join(f"{r.venue}={r.both:.4%}" for r in pd.DataFrame(match_rows).itertuples(index=False)) +
      '; эталон выведен из того же дампа (time_subset 100%) — контроль реализации, см. гейт (a)')
print('(b) no look-ahead : prefix-тест ' +
      ', '.join(f"{r.venue}: {r.mismatches} расхождений из {r.cuts}" for r in pd.DataFrame(prefix_rows).itertuples(index=False)))
gap_tot = sum(BOOKS[n]['S']['gap'] for n in SRC)
gap_b1 = sum(1 for e in BOOKS['bybit']['gap_ev'] if e[1] == 1)
print(f"(c) sequencing    : сбросов {sum(BOOKS[n]['S']['reset'] for n in SRC)}, "
      f"дубликатов {sum(BOOKS[n]['S']['dup_msg'] for n in SRC)}, "
      f"нарушений монотонности {sum(BOOKS[n]['S']['time_back'] for n in SRC)}, "
      f"разрывов {gap_tot} (Bybit: feed1 {gap_b1}, feed2 {gap_tot - gap_b1}; "
      f"осознанный FAIL инварианта no_unrecovered_gap — см. таблицу инвариантов); "
      f"кросс-биржевые джойны as-of backward по event_time")
print(f'(d) reproducible  : хеши состояний совпали на двух прогонах '
      f'{all(run_a[n] == run_b[n] for n in SRC)}')
print()
print('fault injection: поймано', f"{int(CT['caught'].eq('pass').sum())} из {len(CT)}")
print('инварианты уровня сообщения и книги: всего', len(IM) + len(IB),
      '| провалов', int((IM.status == 'FAIL').sum() + (IB.status == 'FAIL').sum()))

СВОДКА ГЕЙТОВ

(a) correct book  : match цен и размеров с эталоном bybit=100.0000%, okx=100.0000%, binance=100.0000%; эталон выведен из того же дампа (time_subset 100%) — контроль реализации, см. гейт (a)
(b) no look-ahead : prefix-тест bybit: 0 расхождений из 24, okx: 0 расхождений из 24, binance: 0 расхождений из 24
(c) sequencing    : сбросов 0, дубликатов 0, нарушений монотонности 0, разрывов 24 (Bybit: feed1 1, feed2 23; осознанный FAIL инварианта no_unrecovered_gap — см. таблицу инвариантов); кросс-биржевые джойны as-of backward по event_time
(d) reproducible  : хеши состояний совпали на двух прогонах True

fault injection: поймано 7 из 7
инварианты уровня сообщения и книги: всего 24 | провалов 1
